In [3]:
import numpy as np

# Define the grid world and parameters
grid_size = 4
num_states = grid_size * grid_size
gamma = 1.0
actions = ['up', 'down', 'right', 'left']
action_probs = [0.25, 0.25, 0.25, 0.25]
reward = -1
terminal_states = [0, 15]  # Representing (0, 0) as 0 and (3, 3) as 15

# Initialize the value function and policy
V = np.zeros(num_states)
policy = {state: np.random.choice(actions) for state in range(num_states) if state not in terminal_states}

def get_next_state(state, action):
    if action == 'up':
        return state - grid_size if state >= grid_size else state
    elif action == 'down':
        return state + grid_size if state < num_states - grid_size else state
    elif action == 'right':
        return state + 1 if (state + 1) % grid_size != 0 else state
    elif action == 'left':
        return state - 1 if state % grid_size != 0 else state

def policy_evaluation(policy, V, theta=1e-6):
    while True:
        delta = 0
        for state in range(num_states):
            if state in terminal_states:
                continue
            v = V[state]
            new_v = 0
            for action, action_prob in zip(actions, action_probs):
                next_state = get_next_state(state, action)
                new_v += action_prob * (reward + gamma * V[next_state])
            V[state] = new_v
            delta = max(delta, abs(v - V[state]))
        if delta < theta:
            break
    return V

def policy_improvement(policy, V):
    policy_stable = True
    for state in range(num_states):
        if state in terminal_states:
            continue
        old_action = policy[state]
        action_values = {}
        for action in actions:
            next_state = get_next_state(state, action)
            action_values[action] = reward + gamma * V[next_state]
        best_action = max(action_values, key=action_values.get)
        policy[state] = best_action
        if best_action != old_action:
            policy_stable = False
    return policy, policy_stable

def policy_iteration(policy, V):
    while True:
        V = policy_evaluation(policy, V)
        policy, policy_stable = policy_improvement(policy, V)
        if policy_stable:
            break
    return policy, V

# Execute policy iteration
optimal_policy, optimal_value_function = policy_iteration(policy, V)

# Display the optimal policy and value function
print("Optimal Policy:")
for state in range(num_states):
    if state in terminal_states:
        print(" T ", end=" ")
    else:
        print(f" {optimal_policy[state][0].upper()} ", end=" ")
    if (state + 1) % grid_size == 0:
        print()

print("\nOptimal Value Function:")
print(optimal_value_function.reshape((grid_size, grid_size)))


Optimal Policy:
 T   L   L   L  
 U   U   L   D  
 U   U   R   D  
 U   R   R   T  

Optimal Value Function:
[[  0.         -13.99999442 -19.99999198 -21.99999117]
 [-13.99999442 -17.99999315 -19.99999257 -19.99999265]
 [-19.99999198 -19.99999257 -17.99999373 -13.99999532]
 [-21.99999117 -19.99999265 -13.99999532   0.        ]]
